# RITA 真假判别推理 (Kaggle)

## 跑之前
1. 右侧 **Settings → Accelerator** 选 `GPU T4 x2` 或 `P100`
2. **Settings → Internet** 打开 (需要 clone GitHub)
3. **Add Input** 挂上两个 Dataset：
   - 数据包：里面是 `dataset.zip`，或已解压的 `<Generator>/<Real|Fake>/<Category>/`
   - 权重包：里面有 `best.pth`

按顺序运行下面每个 cell。先跑试点，确认没问题再跑全量。

In [ ]:
import os, glob, torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0), '| count', torch.cuda.device_count())
print()
print('/kaggle/input 下的内容:')
for d in sorted(os.listdir('/kaggle/input')):
    p = os.path.join('/kaggle/input', d)
    try:
        print(' ', d, '->', sorted(os.listdir(p))[:8])
    except Exception:
        print(' ', d)

In [ ]:
%cd /kaggle/working
REPO = 'https://github.com/你的用户名/你的仓库名.git'   # <-- 改成你的仓库地址
if not os.path.exists('/kaggle/working/RITA'):
    !git clone $REPO RITA
%cd /kaggle/working/RITA
!git log --oneline -3
!ls
!pip install -q timm

In [ ]:
import glob, os
CKPT = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True))
ZIPS = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
DIRS = [p for p in glob.glob('/kaggle/input/**/', recursive=True)
        if os.path.isdir(os.path.join(p, 'ADM'))]
print('权重文件 :', CKPT)
print('压缩包   :', ZIPS)
print('已解压数据根目录:', DIRS)

In [ ]:
import zipfile, os
DATA_ROOT = None
if DIRS:
    DATA_ROOT = DIRS[0]
elif ZIPS:
    DATA_ROOT = '/kaggle/working/data'
    os.makedirs(DATA_ROOT, exist_ok=True)
    print('解压中，约 8GB，需要几分钟...')
    with zipfile.ZipFile(ZIPS[0]) as z:
        z.extractall(DATA_ROOT)
    # zip 内可能还有一层目录，自动下钻
    while not os.path.isdir(os.path.join(DATA_ROOT, 'ADM')):
        subs = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))]
        if not subs:
            break
        DATA_ROOT = os.path.join(DATA_ROOT, subs[0])
else:
    raise RuntimeError('没找到数据，检查 Add Input 里挂的数据集')
print('DATA_ROOT =', DATA_ROOT)
print(sorted(os.listdir(DATA_ROOT))[:20])
assert CKPT, '没找到 .pth 权重文件'

In [ ]:
!cd /kaggle/working/RITA && python -W ignore predict_realfake.py \
    --ckpt {CKPT[0]} \
    --root {DATA_ROOT} \
    --per_gen 20 \
    --batch 8 --workers 2 \
    --out /kaggle/working/preds_pilot.csv \
    --calibrate

---
### 试点结果判读

看 **"每个生成器 AUC"**：
- 平均 AUC 明显高于 0.5（比如 0.7+）→ RITA 在你的生成器域上有信号，值得跑全量
- 平均 AUC 接近 0.5 → RITA 对这些生成器是 OOD 的，融合不会带来增益，别浪费时间跑全量

注意：Real 图在每个生成器下是同一批，整体 ACC 会虚高，**以各生成器 AUC 为准**。

In [ ]:
# 全量推理。7 个完整生成器约 35k 张，GPU 上预计 1~3 小时
!cd /kaggle/working/RITA && python -W ignore predict_realfake.py \
    --ckpt {CKPT[0]} \
    --root {DATA_ROOT} \
    --batch 16 --workers 4 \
    --out /kaggle/working/preds_all.csv

In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/working/preds_pilot.csv')
print(df.shape)
print(df.groupby(['generator','split'])['p_tamper_mean'].describe()[['count','mean','50%']])
df.head()

### 结果下载
右侧 **Output** 面板里可以下载 `preds_pilot.csv` / `preds_all.csv`（在 `/kaggle/working` 下）。

CSV 字段：`generator, split, is_fake, category, path, p_tamper_mean, tamper_ratio, p_tamper_p95, n_steps`

- `p_tamper_mean`：连续分数，算 AUC 用它
- `tamper_ratio`：硬判出的"篡改面积比"，定阈值用它
- `n_steps`：AR 收敛用了多少步，异常值说明序列退化